

## Introduction to Hugging Face

Hugging Face provides **state-of-the-art** NLP (and increasingly, multimodal) models and an easy-to-use Python library called **Transformers**. With just a few lines of code you can:

* **Load** pre-trained models for dozens of tasks
* **Run** inference via high-level “pipelines”
* **Fine-tune** on your own data

---

## 1. Setup in Google Colab

```bash
# Install the core libraries
!pip install transformers datasets huggingface_hub --quiet
```

> **Tip:** Colab often comes with a GPU—go to **Runtime → Change runtime type → GPU** for faster inference.

---

## 2. Quick-start with Pipelines

The `pipeline` API wraps tokenization, model loading, and inference in one object.

### 2.1 Sentiment Analysis

```python
from transformers import pipeline

# 1. Load a sentiment-analysis pipeline (defaults to a small model)
sentiment = pipeline("sentiment-analysis")

# 2. Run inference
examples = [
    "Hugging Face makes NLP super accessible!",
    "I dislike bugs in my code..."
]
results = sentiment(examples)
for text, res in zip(examples, results):
    print(f"{text!r:50} → label={res['label']}, score={res['score']:.3f}")
```

### 2.2 Text Generation

```python
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
prompt = "Once upon a time"
out = generator(prompt, max_length=30, num_return_sequences=1)
print(out[0]["generated_text"])
```

### 2.3 Question Answering

```python
from transformers import pipeline

qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
context = (
    "Hugging Face is an AI company with the mission to democratize good machine learning. "
    "The Transformers library provides thousands of pretrained models in 100+ languages."
)

answer = qa({
    "question": "What is the mission of Hugging Face?",
    "context": context
})
print(answer["answer"])
```

### 2.4 Model Discovery with `huggingface_hub`

```python
from huggingface_hub import HfApi

api = HfApi()
# List top 5 English summarization models
models = api.list_models(task="summarization", limit=5)
for m in models:
    print(m.modelId)
```

---

## 3. Assignment: Build Your Own Pipeline

**Your task:**

1. **Choose** one problem from the list below.
2. **Find** a suitable pre-trained model on [Hugging Face Models](https://huggingface.co/models).
3. **Create** a `pipeline` in Colab to solve the problem—and demonstrate it on 2–3 examples.
4. **Experiment** with model parameters (e.g. `max_length`, `top_k`, `temperature`) and **compare** results.

### 🔹 Problem Set

* **Sentiment classification** of product reviews
* **Text summarization** of news articles
* **Machine translation** (e.g., English ↔ French)
* **Named entity recognition** on a text snippet
* **Paraphrasing** a given sentence
* **Any other** pipeline-supported task you’re curious about!

---



In [14]:
!pip install -q transformers torch

In [15]:
from transformers import pipeline

# **Named Entity Recognition using Hugging Face Pipeline**

Named Entity Recognition means identifying important real-world entities inside text and assigning them categories.

In [25]:
ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

# **Example 1**

In [26]:
text1 = "Elon Musk is the CEO of Tesla and lives in the United States."

results1 = ner_pipeline(text1)

for entity in results1:
    print(
        "Entity:", entity["word"],
        "| Type:", entity["entity_group"],
        "| Confidence:", round(entity["score"], 4)
    )

Entity: El | Type: PER | Confidence: 0.9717
Entity: ##on Musk | Type: PER | Confidence: 0.9373
Entity: Tesla | Type: ORG | Confidence: 0.9706
Entity: United States | Type: LOC | Confidence: 0.9994


# **Example 2**

In [27]:
text2 = "Microsoft announced a new AI research center in London."

results2 = ner_pipeline(text2)

for entity in results2:
    print(
        "Entity:", entity["word"],
        "| Type:", entity["entity_group"],
        "| Confidence:", round(entity["score"], 4)
    )

Entity: Microsoft | Type: ORG | Confidence: 0.9972
Entity: AI | Type: MISC | Confidence: 0.8449
Entity: London | Type: LOC | Confidence: 0.9994


# **Example 3**

In [28]:
text3 = "Abeer attended a technology conference at Superior University in Lahore."

results3 = ner_pipeline(text3)

for entity in results3:
    print(
        "Entity:", entity["word"],
        "| Type:", entity["entity_group"],
        "| Confidence:", round(entity["score"], 4)
    )

Entity: Abe | Type: PER | Confidence: 0.9972
Entity: Superior University | Type: ORG | Confidence: 0.997
Entity: Lahore | Type: LOC | Confidence: 0.999


# **Experiment with parameters**
I am experimenting with the confidence threshold.

In [29]:
text = """
Microsoft CEO Satya Nadella visited Pakistan to discuss
artificial intelligence and technology development.
"""

results = ner_pipeline(text)

for entity in results:
    if entity["score"] > 0.90:
        print(
            entity["word"],
            "→",
            entity["entity_group"],
            "→",
            round(entity["score"], 4)
        )

Microsoft → ORG → 0.9983
Satya Nadella → PER → 0.9261
Pakistan → LOC → 0.9998


# **No change score from 0.90 to 0.70**

In [30]:
text = """
Microsoft CEO Satya Nadella visited Pakistan to discuss
artificial intelligence and technology development.
"""

results = ner_pipeline(text)

for entity in results:
    if entity["score"] > 0.70:
        print(
            entity["word"],
            "→",
            entity["entity_group"],
            "→",
            round(entity["score"], 4)
        )

Microsoft → ORG → 0.9983
Satya Nadella → PER → 0.9261
Pakistan → LOC → 0.9998


## **Conclusion:**
In this experiment, I used a pre-trained BERT-based Named Entity Recognition model through the Hugging Face pipeline. The model successfully identified entities such as people, organizations, and locations from different text snippets and assigned confidence scores to its predictions. I also experimented with the confidence threshold to understand how changing the threshold affects the entities returned by the system. This experiment demonstrated how pre-trained NLP models can extract structured information from unstructured text without requiring us to train a model from scratch.

# Now I specifically wants to check **temperature** and **top_k** parameters so I will use **Text Generation** to experiment these two parameters

In [22]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [23]:
prompt = "Artificial intelligence is changing the world because"

result = generator(
    prompt,
    max_length=50,
    temperature=0.7,
    top_k=50,
    num_return_sequences=1
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial intelligence is changing the world because we have it.

And it's not just us. AI is also changing the way we interact with other people. In a recent paper, authors from MIT, Harvard, and the University of California, Berkeley write that the "social interaction of humans with computers" is changing the way we interact with other people.

"Human interaction with computers is changing the way we interact with other people. They're not just human but are also connected," said Daniel Fried, a postdoctoral fellow in the MIT School of Information Science and Technology.

The paper, which was coauthored by the Max Planck Institute for Machine Intelligence, highlights the link between human interactions with computers and "a human-dominated world."

The paper's lead author, J. Michael Meehan, has long been a researcher at the Harvard School of Information Science and Technology.

"We see this as a huge opportunity for us to find ways to make machines more like humans," Meehan said. "

In [24]:
# now in this cell , I will change the temperture and top_k value to experiment
# temperature: It controls randomness during text generation.
# top_k: It controls the maximum generated sequence length.

prompt = "Artificial intelligence is changing the world because"

result = generator(
    prompt,
    max_length=50,
    temperature=1.2,
    top_k=10,
    num_return_sequences=1
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial intelligence is changing the world because it's so difficult to keep things from moving forward without having a human brain.

"When AI comes to life, we're going to get smarter, more efficient and smarter. The only thing that has to happen is we can learn how to build AI and then it can be used for other things. It won't solve any problem." — Dr. Paul Karp, Stanford professor

What's at stake? The question has the power to change lives. "If you have the ability to develop a human-like intelligence and develop that human-like AI, you've got an opportunity to create the world we need to live in, and we can't just put it out of reach and go out of our minds," explained Dr. Karp.

"It's not an easy question to answer. And that's why the answer is so important. We're going to have to learn how to build and use these systems to create the best human-like technology. But it's going to take a huge amount of work. And the answer lies in technology."

Dr. Karp is one of only a few re

## Conclusion:
The experiment shows that changing temperature and top_k significantly affects the generated text, even though the starting prompt remained the same.

With temperature = 0.7 and top_k = 50, the model produced a more predictable and relatively coherent continuation. After changing to temperature = 1.2 and top_k = 10, the output became more variable and less coherent, with some repetitive and incomplete sentences.

the experiment proves that generation parameters directly influence how a pre-trained language model produces text, and finding suitable parameter values is important for getting useful outputs.